# Bygga bildgenereringsapplikationer (Azure OpenAI)

Det finns mer i LLM än textgenerering. Du kan också generera bilder från textbeskrivningar. Bilder som modalitet är användbara inom MedTech, arkitektur, turism, spelutveckling, marknadsföring och mer. I denna lektion arbetar vi med dagens **GPT Image**-modeller på Microsoft Foundry.

## Lärandemål

I slutet av denna lektion kommer du att kunna:

- Förklara vad bildgenerering är och var det är användbart.
- Förstå `gpt-image` modellfamiljen och hur den skiljer sig från de äldre DALL·E-modellerna.
- Bygga en bildgenereringsapplikation och redigera bilder.

## Vad är bildgenerering?

Bildgenereringsmodeller skapar bilder från en textprompt. Moderna modeller som `gpt-image` lär sig sambandet mellan text och bilder under träningen, och omvandlar sedan iterativt slumpmässigt brus till en bild som matchar din prompt.

Två välkända familjer av bildmodeller är:

- **`gpt-image` (OpenAI)** — den nuvarande generationen som används i denna lektion. Den stödjer text-till-bild-generering och bildredigering (inpainting med mask).
- **Midjourney** — en populär tredjepartsmodell med egen tjänst och Discord-baserat arbetsflöde.

> De äldre OpenAI bildmodellerna — **DALL·E 2** och **DALL·E 3** — är föråldrade. DALL·E 3 är inte längre tillgänglig för nya distributioner, och funktioner som `create_variation` fanns endast i DALL·E 2. Använd `gpt-image`-modellerna för nya applikationer.

På Microsoft Foundry är **`gpt-image-2`** den senaste och mest kapabla bildmodellen och är den rekommenderade som standard. `gpt-image-1.5` och `gpt-image-1-mini` är också allmänt tillgängliga.

> **Viktigt:** `gpt-image` modeller returnerar den genererade bilden som **base64** (`b64_json`), inte som en URL. Din kod avkodar base64-strängen till bytes och sparar den — det finns ingen bild-URL att ladda ner.


## Skapa din första applikation för bildgenerering

Så vad krävs för att skapa en applikation för bildgenerering? Du behöver följande bibliotek:

- **python-dotenv**, det rekommenderas starkt att använda detta bibliotek för att hålla dina hemligheter i en *.env*-fil bort från koden.
- **openai**, detta bibliotek kommer du att använda för att interagera med OpenAI API.
- **pillow**, för att arbeta med bilder i Python.

Om du inte redan gjort det, följ instruktionerna på sidan [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) för att skapa en Microsoft Foundry-resurs och modell. Välj **gpt-image-2** som modell (den senaste Azure OpenAI-bildmodellen; DALL·E är föråldrad).

1. Skapa en fil *.env* med följande innehåll:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Hitta denna information i Microsoft Foundry-portalen för din resurs under sektionen "Deployments".



1. Samla biblioteken ovan i en fil som heter *requirements.txt* så här:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Skapa sedan en virtuell miljö och installera biblioteken:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> För Windows, använd följande kommandon för att skapa och aktivera din virtuella miljö:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Lägg till följande kod i en fil som heter *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # konfigurera Azure OpenAI (Microsoft Foundry) klient.
    # Bildmodeller behöver en aktuell API-version — kontrollera Foundry-dokumentationen för den som din modell kräver.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Skapa en bild med hjälp av API för bildgenerering
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ange din prompt-text här
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # t.ex. "gpt-image-2"
        )
        # Ange katalogen för den sparade bilden
        image_dir = os.path.join(os.curdir, 'images')

        # Om katalogen inte finns, skapa den
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initiera bildvägen (observera att filtypen bör vara png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image-modeller returnerar bilden som base64 (b64_json), inte en URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Visa bilden i standardbildvisaren
        image = Image.open(image_path)
        image.show()

    # fånga undantag
    except BadRequestError as err:
        print(err)

    ```

Låt oss förklara denna kod:

- Först importerar vi biblioteken vi behöver, inklusive OpenAI-biblioteket, dotenv-biblioteket, base64-modulen och Pillow-biblioteket.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Sedan laddar vi miljövariablerna från *.env*-filen.

    ```python
    # importera dotenv
    dotenv.load_dotenv()
    ```

- Därefter konfigurerar vi Azure OpenAI (Microsoft Foundry) klienten.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Sedan genererar vi bilden:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Skriv in din prompttext här
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modeller returnerar bilden som en **base64**-sträng i `data[0].b64_json`. Vi avkodar den till byte och skriver till en fil — det finns ingen URL att ladda ner från.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Slutligen öppnar vi bilden och använder standard bildvisare för att visa den:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Fler detaljer om bildgenerering

Låt oss titta på parametrarna för `images.generate`:

- **prompt**, är textprompten som används för att generera bilden. Här är den "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, är storleken på den genererade bilden (till exempel `1024x1024`, `1536x1024`, `1024x1536` eller `"auto"`).
- **n**, är antalet genererade bilder. Här genererar vi en.
- **model**, är namnet på din bildmodell-implementering (till exempel `gpt-image-2`).

> Bildmodeller tar inte en `temperature`-parameter — det är en kontroll för textgenerering. För att få variation, anropa API:et igen; för att minska variation, gör din prompt mer specifik.

## Ytterligare funktioner vid bildgenerering

Du har sett hur man genererar en bild med några rader Python. `gpt-image` modeller kan också **redigera** en befintlig bild. Genom att tillhandahålla en bild, en valfri **mask** (som markerar området att ändra), och en prompt, kan du ändra en del av bilden — till exempel lägga till en hatt på vår kanin.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# redigeringar returneras också som base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Basbilden innehåller bara kaninen; slutbilden har hatten på kaninen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
